# How LLMs Use Tools

> The same question, asked twice: once to a model on its own, once to a model with tools.

**In this notebook**

1. What tools are, and why models need them
2. The tool loop, written by hand in a few lines of Python
3. Five examples: temperature, temperature + activities, search, calculator, and private data with an action
4. When a model should *not* use a tool

## What are tools?

A **tool** is a function in *your* program that you describe to the model: a name, a purpose and the arguments it takes.

The model never runs anything. It can only reply with a structured request: *"please call `get_temperature` with `city = "Lisbon"`"*. Your code runs the function and sends the result back, and the model then writes its answer using that result.

> **The key idea:** the model decides *what* to call. Your code decides *whether and how* it runs.

## Why do models need tools?

A trained model is a frozen set of weights that predicts the next token. That design leaves four gaps:

| What the model lacks | Why | Tool that fills the gap |
|---|---|---|
| **Fresh information** | Training stopped at a cutoff date | Weather API, web search |
| **Reliable computation** | It predicts digits instead of calculating them | Calculator, code execution |
| **Private data** | Your data was never in its training set | Database or internal API lookup |
| **The ability to act** | Text alone changes nothing in the world | Create a ticket, send an email |

## The tool loop

```text
   ┌────────┐  question + tool descriptions   ┌───────┐
   │  Your  │ ──────────────────────────────▶ │       │
   │  code  │                                 │       │
   │        │ ◀────────────────────────────── │       │
   │        │  "call get_temperature(Lisbon)" │ Model │
   │   ▼    │                                 │       │
   │  runs  │  result: 19.4 °C                │       │
   │  tool  │ ──────────────────────────────▶ │       │
   │        │                                 │       │
   │        │ ◀────────────────────────────── │       │
   └────────┘  "It is 19.4 °C in Lisbon."     └───────┘
```

The middle exchange repeats until the model replies with text instead of a tool call.

## Setup

Needs `OPENAI_API_KEY` in the repo-root `.env`. The optional `OPENAI_MODEL` picks the model. Example 2 also needs `TAVILY_API_KEY` for web search.

In [1]:
import ast
import html
import json
import operator
import os
from datetime import UTC, datetime

import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import HTML, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))

MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.4-mini")
INSTRUCTIONS = "Answer concisely in plain text, without Markdown or LaTeX."
HTTP_HEADERS = {"User-Agent": "intro-ai-notebook/0.1"}

client = OpenAI()
print(f"Model: {MODEL}")

Model: gpt-5.4-mini


### Asking without tools

One request, one answer.

In [2]:
def ask(question):
    response = client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=question,
    )
    return response.output_text

### Asking with tools: the loop

This is the whole mechanism. Agent frameworks add features around it, but the core is this loop.

In [3]:
def ask_with_tools(question, tools, registry, max_turns=6):
    """Returns (answer, trace); registry maps each tool name to its Python function."""
    conversation = [{"role": "user", "content": question}]
    trace = []

    for _ in range(max_turns):
        response = client.responses.create(
            model=MODEL,
            instructions=INSTRUCTIONS,
            input=conversation,
            tools=tools,
        )
        conversation += response.output

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text, trace

        for call in calls:
            arguments = json.loads(call.arguments)
            try:
                result = registry[call.name](**arguments)
            except Exception as error:
                # The model sees the failure and can retry or explain it.
                result = {"error": str(error)}
            trace.append({"tool": call.name, "arguments": arguments, "result": result})
            conversation.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    raise RuntimeError(f"No final answer after {max_turns} turns")


def function_tool(name, description, properties):
    """Describes a function to the model; every property is a required argument."""
    return {
        "type": "function",
        "name": name,
        "description": description,
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": properties,
            "required": list(properties),
            "additionalProperties": False,
        },
    }

### Display helpers

In [4]:
CARD = (
    "font-family:system-ui,sans-serif;font-size:14px;line-height:1.5;color:#1f2933;"
    "background:{bg};border-left:4px solid {accent};border-radius:6px;"
    "padding:10px 14px;margin:8px 0;white-space:pre-wrap"
)


def card(title, body, bg, accent):
    style = CARD.format(bg=bg, accent=accent)
    display(HTML(f'<div style="{style}"><b style="color:{accent}">{html.escape(title)}</b>\n{html.escape(body)}</div>'))


def show_question(question):
    card("Question", question, "#eef2f7", "#52606d")


def show_without_tool(answer):
    card("Without tools", answer, "#fdf1ee", "#c2410c")


def show_with_tool(answer):
    card("With tools", answer, "#ecf8f1", "#15803d")


def show_trace(trace):
    if not trace:
        card("Tool calls", "None: the model answered directly.", "#eef2ff", "#4338ca")
    for step, entry in enumerate(trace, start=1):
        arguments = ", ".join(f"{name}={value!r}" for name, value in entry["arguments"].items())
        result = json.dumps(entry["result"], indent=2, ensure_ascii=False)
        card(f"Step {step}: {entry['tool']}({arguments})", f"returned {result}", "#eef2ff", "#4338ca")

---
# Example 1: Temperature

**Gap: fresh information.** The current temperature changes all the time, so no training set can contain it.

## Example

In [5]:
question = "What is the temperature in Lisbon right now?"
show_question(question)

## Without tool

In [6]:
show_without_tool(ask(question))

## Tools

`get_temperature` is an ordinary Python function that calls the free [Open-Meteo](https://open-meteo.com) API. Nothing about it is specific to AI.

In [7]:
def to_utc_stamp(open_meteo_time):
    return datetime.strptime(open_meteo_time, "%Y-%m-%dT%H:%M").replace(tzinfo=UTC).strftime("%Y-%m-%d-%H-%M-%S")


def get_temperature(city):
    places = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        headers=HTTP_HEADERS,
        timeout=10,
    ).json().get("results")
    if not places:
        return {"error": f"City not found: {city}"}

    place = places[0]
    current = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": place["latitude"], "longitude": place["longitude"], "current": "temperature_2m"},
        headers=HTTP_HEADERS,
        timeout=10,
    ).json()["current"]
    return {
        "city": place["name"],
        "country": place.get("country"),
        "temperature_c": current["temperature_2m"],
        "measured_at_utc": to_utc_stamp(current["time"]),
    }

The **description** is what the model reads. It decides whether and how to call the tool from this text alone; it never sees the Python code.

In [8]:
temperature_tool = function_tool(
    "get_temperature",
    "Get the current air temperature in Celsius for a city. Use for questions about current weather.",
    {"city": {"type": "string", "description": "City name in English, e.g. 'Lisbon'"}},
)

print(json.dumps(temperature_tool, indent=2))

{
  "type": "function",
  "name": "get_temperature",
  "description": "Get the current air temperature in Celsius for a city. Use for questions about current weather.",
  "strict": true,
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "City name in English, e.g. 'Lisbon'"
      }
    },
    "required": [
      "city"
    ],
    "additionalProperties": false
  }
}


## With tool

In [9]:
answer, trace = ask_with_tools(question, [temperature_tool], {"get_temperature": get_temperature})
temperature_run = trace

show_trace(trace)
show_with_tool(answer)

## Explain

- **Without the tool**, the model has no way to know the value. The best it can do is decline or offer a typical figure for the season.
- **With the tool**, the model chose to call `get_temperature` and extracted `city="Lisbon"` from the sentence. Nothing in our code mapped the question to the function.
- **The model did not fetch anything.** Our Python called Open-Meteo. The model only received the JSON and turned it into a sentence.

---
# Example 2: Temperature + activities

**Gap: fresh information, chained.** One tool is often not enough. Here the second tool needs the first tool's result as its input, so the model must run them in order: first find the temperature, then search the web for things to do in that weather.

## Example

In [10]:
question = "I'm in Lisbon today. What can I do there, given the weather right now?"
show_question(question)

## Without tool

In [11]:
show_without_tool(ask(question))

## Tools

`get_temperature` is reused from Example 1. `search_activities` is new: it turns the temperature into a word for the weather (hot, warm, mild, cool, cold) and searches the web with the [Tavily](https://tavily.com) API for things to do in the city in that weather.

Notice the second tool takes `temperature_c` as an argument. Nothing in our code passes it along: the model has to get it from the first tool's result.

In [12]:
WEATHER_BANDS = [(28, "hot"), (20, "warm"), (12, "mild"), (5, "cool"), (float("-inf"), "cold")]


def weather_word(temperature_c):
    return next(word for lowest, word in WEATHER_BANDS if temperature_c >= lowest)


def search_activities(city, temperature_c):
    weather = weather_word(temperature_c)
    query = f"best things to do in {city} in {weather} weather ({temperature_c:.0f}°C)"
    results = requests.post(
        "https://api.tavily.com/search",
        json={"query": query, "max_results": 5},
        headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}", **HTTP_HEADERS},
        timeout=20,
    ).json().get("results", [])
    return {
        "city": city,
        "temperature_c": temperature_c,
        "weather": weather,
        "query": query,
        "results": [{"title": r["title"], "url": r["url"], "snippet": r.get("content", "")[:300]} for r in results],
    }


activities_tool = function_tool(
    "search_activities",
    "Search the web for things to do in a city that suit the current weather. Needs the current temperature, so call get_temperature first.",
    {
        "city": {"type": "string", "description": "City name in English, e.g. 'Lisbon'"},
        "temperature_c": {"type": "number", "description": "Current temperature in Celsius, from get_temperature"},
    },
)

## With tools

In [13]:
answer, trace = ask_with_tools(
    question,
    [temperature_tool, activities_tool],
    {"get_temperature": get_temperature, "search_activities": search_activities},
)
activities_run = trace

show_trace(trace)
show_with_tool(answer)

## Explain

- **Without the tools**, the model doesn't know today's weather, so it can only offer general ideas that might suit any season.
- **With the tools**, the model made two calls in order. The `temperature_c` returned by `get_temperature` became an argument of `search_activities`. Our code never connected the two tools: the model did.
- **The order comes from the descriptions.** `search_activities` says it needs the current temperature, so the model planned two steps. Both calls go through the same small loop, with no new code.
- **The answer combines both results:** the current weather, and activities from the web that suit it.

---
# Example 3: Search

**Gap: fresh information again, but open-ended.** A weather tool answers one kind of question. A search tool lets the model look up almost anything, and it has to write the query itself.

## Example

In [14]:
question = "Who won the 2026 FIFA World Cup, and what was the score in the final?"
show_question(question)

## Without tool

In [15]:
show_without_tool(ask(question))

## Tools

`search` queries Wikipedia and returns the opening paragraphs of the best-matching articles. A production system would often use a web search API, but the idea is the same.

In [16]:
def search(query):
    pages = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "generator": "search",
            "gsrsearch": query,
            "gsrlimit": 3,
            "prop": "extracts",
            "exintro": 1,
            "explaintext": 1,
            "exlimit": 3,
        },
        headers=HTTP_HEADERS,
        timeout=10,
    ).json().get("query", {}).get("pages", {})
    ranked = sorted(pages.values(), key=lambda page: page["index"])
    return [{"title": page["title"], "extract": page.get("extract", "")[:2500]} for page in ranked]


search_tool = function_tool(
    "search",
    "Search Wikipedia. Use for facts that may be newer than your training data. If the results do not answer the question, search again with different keywords. Name the article you used in your answer.",
    {"query": {"type": "string", "description": "A few keywords, e.g. a topic or article title"}},
)

## With tool

In [17]:
answer, trace = ask_with_tools(question, [search_tool], {"search": search})
search_run = trace

show_trace(trace)
show_with_tool(answer)

## Explain

- **Without the tool**, the model cannot answer: the tournament finished after its training data ends. The answer can also be wrongly confident, because a model may believe an event *hasn't happened yet*. Its sense of "now" is frozen at its cutoff.
- **With the tool**, the model wrote its own search query, read the article text and pulled out the two facts it needed: Spain beat Argentina 1–0.
- **The answer is grounded, not remembered.** You can check it against the article in the trace. The quality of the answer now depends on the quality of the source.
- **Descriptions guide, they don't guarantee.** The tool description asks the model to name the article. In this saved run it did, but nothing forces it to on the next run. Anything that must always happen belongs in your code.

---
# Example 4: Calculator

**Gap: reliable computation.** A model generates digits that look likely. It may get a long calculation right or close, but nothing guarantees it. A calculator is exact every time.

## Example

In [18]:
question = (
    "I invest $1,250 at the end of every month at 6.5% annual interest, compounded monthly. "
    "How much do I have after 27 years, to the cent?"
)
show_question(question)

## Without tool

In [19]:
without_tool_answer = ask(question)
show_without_tool(without_tool_answer)

## Tools

`calculate` walks the expression's syntax tree and allows only arithmetic. Python's `eval` would run whatever string the model produced, so it is never used here.

In [20]:
OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}


def evaluate(node):
    match node:
        case ast.Constant(value=int() | float() as value):
            return value
        case ast.BinOp(left=left, op=op, right=right) if type(op) in OPERATORS:
            return OPERATORS[type(op)](evaluate(left), evaluate(right))
        case ast.UnaryOp(op=ast.USub(), operand=operand):
            return -evaluate(operand)
    raise ValueError("Only numbers and + - * / ** are allowed")


def calculate(expression):
    return {"expression": expression, "value": evaluate(ast.parse(expression, mode="eval").body)}


calculator_tool = function_tool(
    "calculate",
    "Evaluate an arithmetic expression exactly. Supports + - * / ** and parentheses. Always use it for arithmetic.",
    {"expression": {"type": "string", "description": "For example '1250 * ((1 + 0.065 / 12) ** 324 - 1)'"}},
)

## With tool

In [21]:
with_tool_answer, trace = ask_with_tools(question, [calculator_tool], {"calculate": calculate})
calculator_run = trace

show_trace(trace)
show_with_tool(with_tool_answer)

### Checking against the formula

The future value of an ordinary annuity, computed directly in Python with no model involved:

In [22]:
monthly_rate = 0.065 / 12
months = 27 * 12
exact = 1250 * ((1 + monthly_rate) ** months - 1) / monthly_rate

card("Exact value", f"${exact:,.2f}", "#eef2f7", "#52606d")

## Explain

- **Without the tool**, the model chose the right formula and showed tidy working, then gave **$744,947.23** in this saved run. That is about **$352,600 too low**. The result sounds exactly as confident as a correct one would, and the figure can change on every run.
- **With the tool**, the model did the part it is good at, turning the words into an expression. Python did the part it is good at, evaluating it. The answer matches the exact value above.
- **This failure is the hardest to notice.** In Examples 1 to 3 the model mostly admits it doesn't know. Here it answers, and only checking reveals the error.

---
# Example 5: Private data and actions

**Gaps: private data and the ability to act.** This example chains two tools: the model looks up an order, reasons about the result, and then changes something.

The order and ticket store below are **fake data**, created for this notebook.

## Example

In [23]:
question = "Where is my order A1042? If it's late, open a support ticket for me."
show_question(question)

## Without tool

In [24]:
show_without_tool(ask(question))

## Tools

One tool **reads** and one tool **writes**. That difference matters: a wrong read gives a bad answer, while a wrong write has real consequences.

In [25]:
ORDERS = {
    "A1042": {
        "item": "Mechanical keyboard",
        "status": "in transit",
        "last_location": "Madrid sorting hub",
        "promised_by_utc": "2026-10-24-18-00-00",
        "days_late": 7,
    },
    "A1043": {
        "item": "USB-C cable",
        "status": "delivered",
        "last_location": "Customer address",
        "promised_by_utc": "2026-10-28-18-00-00",
        "days_late": 0,
    },
}
TICKETS = []


def get_order(order_id):
    return ORDERS.get(order_id, {"error": f"No order {order_id}"})


def create_ticket(order_id, reason):
    ticket = {"ticket_id": f"T-{len(TICKETS) + 1:04d}", "order_id": order_id, "reason": reason}
    TICKETS.append(ticket)
    return ticket


order_tools = [
    function_tool(
        "get_order",
        "Look up an order's status and location. days_late above 0 means it is late.",
        {"order_id": {"type": "string", "description": "Order ID, e.g. 'A1042'"}},
    ),
    function_tool(
        "create_ticket",
        "Open a support ticket about an order. Only use it when the customer asked for one and the condition they gave is met.",
        {
            "order_id": {"type": "string"},
            "reason": {"type": "string", "description": "One sentence for the support team"},
        },
    ),
]
order_registry = {"get_order": get_order, "create_ticket": create_ticket}

## With tool

In [26]:
answer, trace = ask_with_tools(question, order_tools, order_registry)
order_run = trace

show_trace(trace)
show_with_tool(answer)

print("Ticket store after the run:", TICKETS)

Ticket store after the run: [{'ticket_id': 'T-0001', 'order_id': 'A1042', 'reason': 'Order is 7 days late and the customer requested a support ticket.'}]


Now an order that is **not** late, with the same tools and the same request. The condition in the request should stop the second tool.

In [27]:
TICKETS_BEFORE = len(TICKETS)
question = "Where is my order A1043? If it's late, open a support ticket for me."
show_question(question)

answer, trace = ask_with_tools(question, order_tools, order_registry)

show_trace(trace)
show_with_tool(answer)

print("New tickets:", len(TICKETS) - TICKETS_BEFORE)

New tickets: 0


## Explain

- **Without the tools**, the model has no access to the order and can't open a ticket. It can only ask for details and draft a message for you to send.
- **With the tools**, the second step depended on the first. The model read `days_late: 7`, applied your condition, and only then called `create_ticket`. Our code never contained that `if`.
- **The action really happened**: `TICKETS` changed, so this is no longer only text.
- **The second run shows the decision:** order A1043 has `days_late: 0`, so the model stopped after the lookup and no ticket was created.
- **This is also the start of an agent:** a model in a loop, choosing its next step from what the previous step returned.

---
# When not to use a tool

Tools are options, not obligations. Here the model gets **every tool** from this notebook and a question it already knows the answer to.

In [28]:
all_tools = [temperature_tool, activities_tool, search_tool, calculator_tool, *order_tools]
all_registry = {
    "get_temperature": get_temperature,
    "search_activities": search_activities,
    "search": search,
    "calculate": calculate,
    **order_registry,
}

question = "What is the capital of France?"
show_question(question)

answer, trace = ask_with_tools(question, all_tools, all_registry)

show_trace(trace)
show_with_tool(answer)

With six tools available, the model made **no calls** and answered directly. Offering a tool gives the model a choice, and each description tells it when that choice applies.

---
# Summary

In [29]:
def tools_used(run):
    return " → ".join(entry["tool"] for entry in run)


rows = [
    ("Temperature", "Fresh information", temperature_run),
    ("Temperature + activities", "Fresh information, chained", activities_run),
    ("Search", "Fresh information", search_run),
    ("Calculator", "Reliable computation", calculator_run),
    ("Order lookup + ticket", "Private data, actions", order_run),
]
cell = "padding:6px 12px;border-bottom:1px solid #cbd2d9;text-align:left"
header = "".join(f'<th style="{cell}">{title}</th>' for title in ("Example", "Gap filled", "Tool calls in this run"))
body = "".join(
    f'<tr><td style="{cell}">{name}</td><td style="{cell}">{gap}</td><td style="{cell}"><code>{tools_used(run)}</code></td></tr>'
    for name, gap, run in rows
)
display(HTML(f'<table style="border-collapse:collapse;font-family:system-ui,sans-serif"><tr>{header}</tr>{body}</table>'))

Example,Gap filled,Tool calls in this run
Temperature,Fresh information,get_temperature
Temperature + activities,"Fresh information, chained",get_temperature → search_activities
Search,Fresh information,search
Calculator,Reliable computation,calculate
Order lookup + ticket,"Private data, actions",get_order → create_ticket


## Takeaways

1. **The model requests, your code executes.** A tool call is structured output, not execution.
2. **The loop is small.** Send the question and tools, run what the model asks for, send the results back, and repeat until it answers in text.
3. **Descriptions are prompts.** The model chooses tools from their names and descriptions, so write them with care.
4. **Tool arguments are untrusted input.** That is why `calculate` does not use `eval`. Validate arguments as you would for any external request.
5. **Tool results are untrusted too.** A web page returned by `search` could contain text that tries to give the model instructions, so results must not be able to unlock dangerous actions.
6. **Separate reads from writes.** For tools that change state, add permission checks or human confirmation in your code, not only in the description.